# Saccade-aligned spike rate pipeline

**Environment:** Use the `eye_repo` conda environment.

This pipeline loads the exported `all_saccade_collection`, infers the required (animal, block) set, loads Kilosort spike data per block, and computes **average spike rate** (spikes per second) in a **3 s window** around saccade onset (t = 0). Run cells **top to bottom** in order.

**User input:** Set paths and `output_folder` in the **Config** cell; all plots are exported there as high-quality PDFs.

**Prerequisites:**
- Run `saccade_collection_pipeline.ipynb` and export `all_saccade_collection` to the configured `export_dir`.
- Kilosort output available per block.
- BlockSync blocks with `oe_rec` initialized.


In [ ]:
# =============================================================================
# IMPORTS AND DEFINITIONS — run this cell first
# =============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

from eye_tracking_system_tools.preprocessing import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.kilosort_loader import (
    load_and_align_kilosort_spikes,
    get_good_cluster_ids,
)


def _block_key(animal, block):
    b = str(block).strip()
    if len(b) < 3:
        b = b.zfill(3)
    return f"{animal}_block_{b}"


def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    if bad_blocks is None:
        bad_blocks = []
    block_collection = []
    block_dict = {}
    for animal, blocks in zip(animals, block_lists):
        current = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks,
        )
        block_collection.extend(current)
        for b in current:
            block_dict[f"{animal}_block_{b.block_num}"] = b
    return block_collection, block_dict


def extract_mean_spike_rate_per_animal(
    collection_df,
    block_dict,
    window_ms,
    half_window_ms,
    bin_ms,
    animals,
    cluster_ids=None,
    kilosort_path=None,
    filter_good_only=True,
    desc_prefix="Spike rate",
):
    """
    Compute average spike rate (spikes/s) in bins around saccade onset, per animal.
    Returns dict animal -> (t_rel_ms, rate_hz, n_saccades).
    """
    n_bins = int(window_ms / bin_ms)
    bin_edges_rel_ms = np.linspace(-half_window_ms, half_window_ms, n_bins + 1)
    bin_center_rel_ms = (bin_edges_rel_ms[:-1] + bin_edges_rel_ms[1:]) / 2
    bin_duration_s = bin_ms / 1000.0

    mean_rate_per_animal = {}
    for animal in animals:
        rows = collection_df[collection_df["animal"] == animal]
        if rows.empty:
            continue

        block_groups = defaultdict(list)
        for _, r in rows.iterrows():
            key = _block_key(r["animal"], r["block"])
            block = block_dict.get(key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            onset_ms = float(r["saccade_on_ms"])
            start_ms = onset_ms - half_window_ms
            end_ms = onset_ms + half_window_ms
            dur_ms = getattr(block.oe_rec, "recordingDuration_ms", None)
            if dur_ms is not None:
                dur_ms = float(dur_ms) if not hasattr(dur_ms, "item") else float(dur_ms.item())
            if dur_ms is not None and (start_ms < 0 or end_ms > dur_ms):
                continue
            block_groups[key].append((onset_ms, r))

        if not block_groups:
            continue

        all_bin_counts = []
        pbar = tqdm(block_groups.items(), desc=f"{desc_prefix} {animal}", leave=False)
        for block_key, saccade_list in pbar:
            block = block_dict.get(block_key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            try:
                spike_times_ms, spike_clusters, meta = load_and_align_kilosort_spikes(
                    block, kilosort_path=kilosort_path, filter_good_only=filter_good_only
                )
            except Exception as e:
                pbar.set_postfix({"error": str(e)[:20]})
                continue

            if cluster_ids is not None:
                cids = np.atleast_1d(cluster_ids)
                mask = np.isin(spike_clusters, cids)
                spike_times_ms = spike_times_ms[mask]

            for onset_ms, _ in saccade_list:
                rel_ms = spike_times_ms - onset_ms
                in_window = (rel_ms >= -half_window_ms) & (rel_ms < half_window_ms)
                rel_in = rel_ms[in_window]
                counts, _ = np.histogram(rel_in, bins=bin_edges_rel_ms)
                all_bin_counts.append(counts.astype(float))

            pbar.set_postfix({"n_sacc": len(saccade_list), "valid": len(all_bin_counts)})

        if not all_bin_counts:
            continue

        stack = np.stack(all_bin_counts)
        mean_counts = np.nanmean(stack, axis=0)
        rate_hz = mean_counts / bin_duration_s
        n_saccades = len(all_bin_counts)
        mean_rate_per_animal[animal] = (bin_center_rel_ms, rate_hz, n_saccades)

    return mean_rate_per_animal


def extract_mean_spike_rate_per_unit(
    collection_df,
    block_dict,
    window_ms,
    half_window_ms,
    bin_ms,
    animals,
    kilosort_path=None,
    filter_good_only=True,
    desc_prefix="Per-unit rate",
):
    """
    Compute average spike rate (spikes/s) per Phy/Kilosort unit in bins around saccade onset.
    Returns dict animal -> dict cluster_id -> (t_rel_ms, rate_hz, n_saccades).
    """
    n_bins = int(window_ms / bin_ms)
    bin_edges_rel_ms = np.linspace(-half_window_ms, half_window_ms, n_bins + 1)
    bin_center_rel_ms = (bin_edges_rel_ms[:-1] + bin_edges_rel_ms[1:]) / 2
    bin_duration_s = bin_ms / 1000.0

    mean_rate_per_animal_per_unit = {}
    for animal in animals:
        rows = collection_df[collection_df["animal"] == animal]
        if rows.empty:
            continue

        block_groups = defaultdict(list)
        for _, r in rows.iterrows():
            key = _block_key(r["animal"], r["block"])
            block = block_dict.get(key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            onset_ms = float(r["saccade_on_ms"])
            start_ms = onset_ms - half_window_ms
            end_ms = onset_ms + half_window_ms
            dur_ms = getattr(block.oe_rec, "recordingDuration_ms", None)
            if dur_ms is not None:
                dur_ms = float(dur_ms) if not hasattr(dur_ms, "item") else float(dur_ms.item())
            if dur_ms is not None and (start_ms < 0 or end_ms > dur_ms):
                continue
            block_groups[key].append((onset_ms, r))

        if not block_groups:
            continue

        cluster_bin_counts = defaultdict(list)
        pbar = tqdm(block_groups.items(), desc=f"{desc_prefix} {animal}", leave=False)

        for block_key, saccade_list in pbar:
            block = block_dict.get(block_key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            try:
                spike_times_ms, spike_clusters, meta = load_and_align_kilosort_spikes(
                    block, kilosort_path=kilosort_path, filter_good_only=filter_good_only
                )
            except Exception as e:
                pbar.set_postfix({"error": str(e)[:20]})
                continue

            unique_clusters = np.unique(spike_clusters)
            for cluster_id in unique_clusters:
                mask = spike_clusters == cluster_id
                st_ms = spike_times_ms[mask]
                for onset_ms, _ in saccade_list:
                    rel_ms = st_ms - onset_ms
                    in_window = (rel_ms >= -half_window_ms) & (rel_ms < half_window_ms)
                    rel_in = rel_ms[in_window]
                    counts, _ = np.histogram(rel_in, bins=bin_edges_rel_ms)
                    cluster_bin_counts[int(cluster_id)].append(counts.astype(float))

            pbar.set_postfix({"clusters": len(cluster_bin_counts), "sacc": len(saccade_list)})

        if not cluster_bin_counts:
            continue

        rate_per_unit = {}
        for cid, count_list in cluster_bin_counts.items():
            if not count_list:
                continue
            stack = np.stack(count_list)
            mean_counts = np.nanmean(stack, axis=0)
            rate_hz = mean_counts / bin_duration_s
            rate_per_unit[cid] = (bin_center_rel_ms.copy(), rate_hz, len(count_list))

        mean_rate_per_animal_per_unit[animal] = rate_per_unit

    return mean_rate_per_animal_per_unit


def extract_spike_times_for_raster(
    collection_df,
    block_dict,
    window_ms,
    half_window_ms,
    animals,
    cluster_ids=None,
    kilosort_path=None,
    filter_good_only=True,
    max_saccades_per_animal=500,
    desc_prefix="Raster",
):
    """
    Extract spike times relative to saccade onset for raster plots.
    Returns dict animal -> list of (saccade_idx, spike_times_rel_ms) tuples.
    """
    spike_times_per_animal = {}

    for animal in animals:
        rows = collection_df[collection_df["animal"] == animal].copy()
        if rows.empty:
            continue

        if len(rows) > max_saccades_per_animal:
            rows = rows.sample(n=max_saccades_per_animal, random_state=42).sort_index()

        block_groups = defaultdict(list)
        for idx, (_, r) in enumerate(rows.iterrows()):
            key = _block_key(r["animal"], r["block"])
            block = block_dict.get(key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            onset_ms = float(r["saccade_on_ms"])
            start_ms = onset_ms - half_window_ms
            end_ms = onset_ms + half_window_ms
            dur_ms = getattr(block.oe_rec, "recordingDuration_ms", None)
            if dur_ms is not None:
                dur_ms = float(dur_ms) if not hasattr(dur_ms, "item") else float(dur_ms.item())
            if dur_ms is not None and (start_ms < 0 or end_ms > dur_ms):
                continue
            block_groups[key].append((idx, onset_ms, r))

        if not block_groups:
            continue

        saccade_spike_times = []
        pbar = tqdm(block_groups.items(), desc=f"{desc_prefix} {animal}", leave=False)

        for block_key, saccade_list in pbar:
            block = block_dict.get(block_key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue

            try:
                spike_times_ms, spike_clusters, meta = load_and_align_kilosort_spikes(
                    block, kilosort_path=kilosort_path, filter_good_only=filter_good_only
                )
            except Exception as e:
                pbar.set_postfix({"error": str(e)[:20]})
                continue

            if cluster_ids is not None:
                cids = np.atleast_1d(cluster_ids)
                mask = np.isin(spike_clusters, cids)
                spike_times_ms = spike_times_ms[mask]

            for saccade_idx, onset_ms, _ in saccade_list:
                rel_ms = spike_times_ms - onset_ms
                in_window = (rel_ms >= -half_window_ms) & (rel_ms < half_window_ms)
                spike_times_rel = rel_ms[in_window]
                saccade_spike_times.append((saccade_idx, spike_times_rel))

            pbar.set_postfix({"n_sacc": len(saccade_list), "valid": len(saccade_spike_times)})

        if saccade_spike_times:
            saccade_spike_times.sort(key=lambda x: x[0])
            spike_times_per_animal[animal] = saccade_spike_times

    return spike_times_per_animal


def extract_pooled_rate_matrix_per_saccade(
    collection_df,
    block_dict,
    window_ms,
    half_window_ms,
    bin_ms,
    animals,
    cluster_ids=None,
    kilosort_path=None,
    filter_good_only=True,
    desc_prefix="Per-saccade rate",
):
    """
    Compute per-saccade pooled spike rate (spikes/s) in time bins.
    Returns dict animal -> (t_rel_ms, rate_matrix, magnitudes) where rate_matrix is (n_saccades, n_bins)
    and magnitudes is (n_saccades,) for ordering (e.g. by saccade amplitude). Uses "magnitude" column if present.
    """
    n_bins = int(window_ms / bin_ms)
    bin_edges_rel_ms = np.linspace(-half_window_ms, half_window_ms, n_bins + 1)
    bin_center_rel_ms = (bin_edges_rel_ms[:-1] + bin_edges_rel_ms[1:]) / 2
    bin_duration_s = bin_ms / 1000.0

    result_per_animal = {}
    for animal in animals:
        rows = collection_df[collection_df["animal"] == animal]
        if rows.empty:
            continue

        block_groups = defaultdict(list)
        for _, r in rows.iterrows():
            key = _block_key(r["animal"], r["block"])
            block = block_dict.get(key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            onset_ms = float(r["saccade_on_ms"])
            start_ms = onset_ms - half_window_ms
            end_ms = onset_ms + half_window_ms
            dur_ms = getattr(block.oe_rec, "recordingDuration_ms", None)
            if dur_ms is not None:
                dur_ms = float(dur_ms) if not hasattr(dur_ms, "item") else float(dur_ms.item())
            if dur_ms is not None and (start_ms < 0 or end_ms > dur_ms):
                continue
            block_groups[key].append((onset_ms, r))

        if not block_groups:
            continue

        all_bin_counts = []
        magnitudes_list = []
        mag_col = "magnitude" if "magnitude" in collection_df.columns else None
        pbar = tqdm(block_groups.items(), desc=f"{desc_prefix} {animal}", leave=False)
        for block_key, saccade_list in pbar:
            block = block_dict.get(block_key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            try:
                spike_times_ms, spike_clusters, meta = load_and_align_kilosort_spikes(
                    block, kilosort_path=kilosort_path, filter_good_only=filter_good_only
                )
            except Exception as e:
                pbar.set_postfix({"error": str(e)[:20]})
                continue

            if cluster_ids is not None:
                cids = np.atleast_1d(cluster_ids)
                mask = np.isin(spike_clusters, cids)
                spike_times_ms = spike_times_ms[mask]

            for onset_ms, r in saccade_list:
                rel_ms = spike_times_ms - onset_ms
                in_window = (rel_ms >= -half_window_ms) & (rel_ms < half_window_ms)
                rel_in = rel_ms[in_window]
                counts, _ = np.histogram(rel_in, bins=bin_edges_rel_ms)
                all_bin_counts.append(counts.astype(float))
                try:
                    mag = float(r.get(mag_col, np.nan)) if mag_col else np.nan
                except (TypeError, ValueError):
                    mag = np.nan
                magnitudes_list.append(np.nan if (mag != mag) else mag)

            pbar.set_postfix({"n_sacc": len(saccade_list), "rows": len(all_bin_counts)})

        if not all_bin_counts:
            continue

        stack = np.stack(all_bin_counts)
        rate_hz = stack / bin_duration_s
        mag_arr = np.array(magnitudes_list, dtype=float)
        result_per_animal[animal] = (bin_center_rel_ms, rate_hz, mag_arr)

    return result_per_animal


def _sanitize(s):
    """Sanitize condition/animal name for use in filenames."""
    return "".join(c if c.isalnum() or c in " _-" else "_" for c in s).strip("_") or "plot"

def load_eye_data(block):
    """
    Load the eye dataframes from CSV files created by the synchronization pipeline.
    No rotation matrices are loaded as rotation is no longer used.
    :param block: The current blocksync class
    :return: None
    """
    try:
        block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data_degrees_raw_verified.csv', index_col=0, engine='python')
        block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data_degrees_raw_verified.csv', index_col=0, engine='python')
        print(f'Loaded eye data for block {block.block_num}')
    except FileNotFoundError:
        print('Eye data files not found. Run the synchronization pipeline first!')
        raise


## User input: config

Set `experiment_path`, `collection_path`, window and bin size, `cluster_ids`, and **`output_folder`** (all PDFs are saved there).


In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")
export_dir = experiment_path / "analysis" / "saccade_collections"
export_filename = "all_saccade_collection.csv"
collection_path = export_dir / export_filename

window_ms = 6000.0   # 3 s total around saccade onset
half_window_ms = window_ms / 2
bin_ms = 50.0        # bin size for rate (ms)

# Which clusters to include: None = pool all good clusters; or list e.g. [0, 1, 2]
cluster_ids = None

# Output folder for all PDF plots (high-quality export). Create if missing.
output_folder = Path(export_dir) / "spike_rate_plots_longer"
output_folder.mkdir(parents=True, exist_ok=True)



## Load `all_saccade_collection` and infer (animal, block) set


In [ ]:
all_saccade_collection = pd.read_csv(collection_path)
required = ["animal", "block", "Main", "Sub", "saccade_on_ms"]
missing = [c for c in required if c not in all_saccade_collection.columns]
if missing:
    raise ValueError(f"all_saccade_collection missing columns: {missing}")
print(f"Loaded {len(all_saccade_collection):,} rows from {collection_path.name}")

ab = all_saccade_collection[["animal", "block"]].drop_duplicates()
animals = ab["animal"].dropna().unique().astype(str).tolist()
block_lists = [
    [int(b) for b in ab.loc[ab["animal"] == a, "block"].dropna().unique().astype(str).tolist()]
    for a in animals
]
print(f"Inferred animals: {animals}; block_lists: {block_lists}")



## Build `block_collection` and `block_dict`


In [ ]:
block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=[],
)
print(f"Blocks: {list(block_dict.keys())}")
for block in block_collection:
    load_eye_data(block)



## Filter saccades into sub-categories


In [ ]:
if "head_movement" in all_saccade_collection.columns:
    mask_no_head = ~all_saccade_collection["head_movement"].fillna(False).astype(bool)
    mask_with_head = np.ones(len(all_saccade_collection), dtype=bool)
else:
    mask_no_head = np.ones(len(all_saccade_collection), dtype=bool)
    mask_with_head = np.ones(len(all_saccade_collection), dtype=bool)
    print("Note: 'head_movement' column not in collection.")

synced = all_saccade_collection.dropna(subset=["Main"]).copy()
synced_one_per_pair = synced.query('Sub == "L"').copy()
synced_with_head = synced_one_per_pair.copy()
synced_no_head = synced_one_per_pair.loc[synced_one_per_pair.index.intersection(all_saccade_collection.index[mask_no_head])].copy()

mono_left = all_saccade_collection[all_saccade_collection["Main"].isna() & (all_saccade_collection["eye"] == "L")].copy()
mono_left_with_head = mono_left.copy()
mono_left_no_head = mono_left.loc[mono_left.index.intersection(all_saccade_collection.index[mask_no_head])].copy()

mono_right = all_saccade_collection[all_saccade_collection["Main"].isna() & (all_saccade_collection["eye"] == "R")].copy()
mono_right_with_head = mono_right.copy()
mono_right_no_head = mono_right.loc[mono_right.index.intersection(all_saccade_collection.index[mask_no_head])].copy()

CONJUGATED_DEG = 45
synced_l_full = synced.query('Sub == "L"').copy()
if "angle" in synced.columns:
    synced_l = synced.query('Sub == "L"')[["Main", "animal", "block", "angle", "saccade_on_ms"]].rename(columns={"angle": "angle_L"})
    synced_r = synced.query('Sub == "R"')[["Main", "angle"]].rename(columns={"angle": "angle_R"})
    pair_angles = synced_l.merge(synced_r, on="Main", how="inner")
    pair_angles["angle_diff"] = np.abs(pair_angles["angle_L"] - pair_angles["angle_R"])
    pair_angles["angle_diff"] = np.minimum(pair_angles["angle_diff"], 360 - pair_angles["angle_diff"])
    pair_angles["conjugated"] = pair_angles["angle_diff"] <= CONJUGATED_DEG
    pair_angles = pair_angles.set_index("Main")
    conj_main_ids = pair_angles.index[pair_angles["conjugated"]].values
    non_conj_main_ids = pair_angles.index[~pair_angles["conjugated"]].values
    conjugated_rows = synced_l_full[synced_l_full["Main"].isin(conj_main_ids)].copy()
    non_conjugated_rows = synced_l_full[synced_l_full["Main"].isin(non_conj_main_ids)].copy()
else:
    conjugated_rows = pd.DataFrame()
    non_conjugated_rows = pd.DataFrame()

conjugated_with_head = conjugated_rows.copy()
conjugated_no_head = conjugated_rows.loc[conjugated_rows.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
non_conjugated_with_head = non_conjugated_rows.copy()
non_conjugated_no_head = non_conjugated_rows.loc[non_conjugated_rows.index.intersection(all_saccade_collection.index[mask_no_head])].copy()

filtered_collections = [
    ("Synced (with head)", synced_with_head),
    ("Synced (no head)", synced_no_head),
    ("Monocular left (with head)", mono_left_with_head),
    ("Monocular left (no head)", mono_left_no_head),
    ("Monocular right (with head)", mono_right_with_head),
    ("Monocular right (no head)", mono_right_no_head),
    ("Conjugated (with head)", conjugated_with_head),
    ("Conjugated (no head)", conjugated_no_head),
    ("Non-conjugated (with head)", non_conjugated_with_head),
    ("Non-conjugated (no head)", non_conjugated_no_head),
]



## Compute mean spike rate per filtered collection


In [ ]:
mean_rate_by_filter = {}
for name, df in tqdm(filtered_collections, desc="Filtered collections"):
    if df.empty:
        mean_rate_by_filter[name] = {}
        continue
    mean_rate_by_filter[name] = extract_mean_spike_rate_per_animal(
        df,
        block_dict,
        window_ms,
        half_window_ms,
        bin_ms,
        animals,
        cluster_ids=cluster_ids,
        kilosort_path=None,
        filter_good_only=True,
        desc_prefix=f"Rate {name[:15]}",
    )
    n_sacc = sum(
        mean_rate_by_filter[name][a][2] if a in mean_rate_by_filter[name] else 0
        for a in animals
    )
    print(f"{name}: animals with rate={len(mean_rate_by_filter[name])}, total saccades used={n_sacc}")



## Whole-recording explorable: spike rate + eye traces (Bokeh)

Simple whole-recording trace plot: **left and right eye (centered)** k_phi, k_theta, pupil_diameter and **rolling-window average spike rate** (500 ms window, 100 ms step). All traces share one explorable time axis (pan/zoom linked via Bokeh). Pick `explorable_block_key` below (e.g. `"PV_126_block_006"`) or leave `None` to use the first block.

In [ ]:
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.layouts import column as bokeh_column

output_notebook()

# Which block to plot (None = first block in block_dict)
explorable_block_key = "PV_208_block_019"  # e.g. "PV_126_block_006"
if explorable_block_key is None:
    explorable_block_key = next(iter(block_dict.keys()))
block = block_dict.get(explorable_block_key)
if block is None:
    raise ValueError(f"Block {explorable_block_key} not in block_dict.")

# Eye data: prefer centered (k_phi_recentered, k_theta_recentered), else k_phi, k_theta; pupil_diameter
left_df = getattr(block, "left_eye_data_centered", None) or getattr(block, "left_eye_data", None)
right_df = getattr(block, "right_eye_data_centered", None) or getattr(block, "right_eye_data", None)
if left_df is None and right_df is None:
    raise ValueError(f"No eye data on block {explorable_block_key}. Need left_eye_data/right_eye_data or left_eye_data_centered/right_eye_data_centered.")

def _phi_col(df):
    return "k_phi_recentered" if "k_phi_recentered" in df.columns else "k_phi" if "k_phi" in df.columns else "phi"
def _theta_col(df):
    return "k_theta_recentered" if "k_theta_recentered" in df.columns else "k_theta" if "k_theta" in df.columns else "theta"
def _pupil_col(df):
    return "pupil_diameter" if "pupil_diameter" in df.columns else "major_ax"

# Rolling spike rate: 500 ms window, 100 ms step
roll_window_ms = 500.0
roll_step_ms = 100.0
spike_times_ms, spike_clusters, _ = load_and_align_kilosort_spikes(block, kilosort_path=None, filter_good_only=True)
dur_ms = float(getattr(block.oe_rec, "recordingDuration_ms", spike_times_ms.max() + 1))
if hasattr(dur_ms, "item"):
    dur_ms = float(dur_ms.item())
t_centers = np.arange(roll_window_ms / 2, dur_ms - roll_window_ms / 2, roll_step_ms)
half = roll_window_ms / 2
rates = np.array([np.sum((spike_times_ms >= t - half) & (spike_times_ms < t + half)) / (roll_window_ms / 1000.0) for t in t_centers])

# Shared time axis for linked zoom (use full recording span)
t_min = 0.0
t_max = max(dur_ms, t_centers.max() + half if len(t_centers) else dur_ms)
if left_df is not None and len(left_df):
    t_max = max(t_max, left_df["ms_axis"].max())
if right_df is not None and len(right_df):
    t_max = max(t_max, right_df["ms_axis"].max())
x_range = (t_min, t_max)
tools = "pan,wheel_zoom,box_zoom,reset,save"
fig_kw = dict(width=900, height=180, x_axis_label="Time (ms)", tools=tools)

plots = []
# φ (phi): L and R (first figure sets shared x_range)
p_phi = figure(title="φ (phi) — L & R", x_range=x_range, **fig_kw)
p_phi.yaxis.axis_label = "φ (°)"
if left_df is not None and "ms_axis" in left_df.columns:
    c = _phi_col(left_df)
    p_phi.line(left_df["ms_axis"].values, pd.to_numeric(left_df[c], errors="coerce"), line_width=1.2, color="crimson", legend_label="L")
if right_df is not None and "ms_axis" in right_df.columns:
    c = _phi_col(right_df)
    p_phi.line(right_df["ms_axis"].values, pd.to_numeric(right_df[c], errors="coerce"), line_width=1.2, color="steelblue", legend_label="R")
plots.append(p_phi)

# θ (theta): L and R
p_theta = figure(title="θ (theta) — L & R", x_range=p_phi.x_range, **fig_kw)
p_theta.yaxis.axis_label = "θ (°)"
if left_df is not None and "ms_axis" in left_df.columns:
    c = _theta_col(left_df)
    p_theta.line(left_df["ms_axis"].values, pd.to_numeric(left_df[c], errors="coerce"), line_width=1.2, color="crimson", legend_label="L")
if right_df is not None and "ms_axis" in right_df.columns:
    c = _theta_col(right_df)
    p_theta.line(right_df["ms_axis"].values, pd.to_numeric(right_df[c], errors="coerce"), line_width=1.2, color="steelblue", legend_label="R")
plots.append(p_theta)

# Pupil diameter: L and R
p_pupil = figure(title="Pupil diameter — L & R", x_range=p_phi.x_range, **fig_kw)
p_pupil.yaxis.axis_label = "Pupil (a.u.)"
if left_df is not None and "ms_axis" in left_df.columns:
    c = _pupil_col(left_df)
    p_pupil.line(left_df["ms_axis"].values, pd.to_numeric(left_df[c], errors="coerce"), line_width=1.2, color="crimson", legend_label="L")
if right_df is not None and "ms_axis" in right_df.columns:
    c = _pupil_col(right_df)
    p_pupil.line(right_df["ms_axis"].values, pd.to_numeric(right_df[c], errors="coerce"), line_width=1.2, color="steelblue", legend_label="R")
plots.append(p_pupil)

# Rolling spike rate (500 ms window, 100 ms step)
p_rate = figure(title=f"Spike rate (Hz) — {roll_window_ms:.0f} ms window, {roll_step_ms:.0f} ms step", x_range=p_phi.x_range, **fig_kw)
p_rate.yaxis.axis_label = "Rate (Hz)"
p_rate.line(t_centers, rates, line_width=1.5, color="black")
plots.append(p_rate)

show(bokeh_column(*plots))

*(Whole-recording explorable plot is in the section above.)*

In [ ]:
# Whole-recording explorable plot is in the cell above (spike rate + eye traces, linked time axis).

## Extract mean eye traces (for explorable plot)

Extract saccade-aligned mean eye traces (φ, θ, pupil) per animal for the same filtered collections, so we can plot them together with spike rate in an explorable Bokeh figure. Requires blocks to have `left_eye_data` / `right_eye_data` (or CSV under `analysis_path`). Uses `k_phi`/`phi`, `k_theta`/`theta`, and `pupil_diameter` or `major_ax` for pupil.

## Compute mean spike rate per unit


In [ ]:
rate_per_unit_by_filter = {}
for name, df in tqdm(filtered_collections, desc="Per-unit rate extraction"):
    if df.empty:
        rate_per_unit_by_filter[name] = {}
        continue
    rate_per_unit_by_filter[name] = extract_mean_spike_rate_per_unit(
        df,
        block_dict,
        window_ms,
        half_window_ms,
        bin_ms,
        animals,
        kilosort_path=None,
        filter_good_only=True,
        desc_prefix=f"Unit {name[:12]}",
    )
    n_animals = len(rate_per_unit_by_filter[name])
    n_units = sum(len(rate_per_unit_by_filter[name].get(a, {})) for a in animals)
    print(f"{name}: animals={n_animals}, total units={n_units}")



## Compute per-saccade rate matrix (for heatmap)


In [ ]:
per_saccade_rate_by_filter = {}
for name, df in tqdm(filtered_collections, desc="Per-saccade rate matrix"):
    if df.empty:
        per_saccade_rate_by_filter[name] = {}
        continue
    per_saccade_rate_by_filter[name] = extract_pooled_rate_matrix_per_saccade(
        df,
        block_dict,
        window_ms,
        half_window_ms,
        bin_ms,
        animals,
        cluster_ids=cluster_ids,
        kilosort_path=None,
        filter_good_only=True,
        desc_prefix=f"Heatmap {name[:12]}",
    )
    n_sacc = sum(
        per_saccade_rate_by_filter[name][a][1].shape[0] if a in per_saccade_rate_by_filter[name] else 0
        for a in animals
    )
    print(f"{name}: total saccades in matrix={n_sacc}")

# Magnitude column for ordering (used in raster and heatmap)
magnitude_col = "magnitude" if "magnitude" in all_saccade_collection.columns else None



## Compute raster data per filtered collection


In [ ]:
raster_data_by_filter = {}
max_raster_saccades = 500

for name, df in tqdm(filtered_collections, desc="Raster data extraction"):
    if df.empty:
        raster_data_by_filter[name] = {}
        continue
    raster_data_by_filter[name] = extract_spike_times_for_raster(
        df,
        block_dict,
        window_ms,
        half_window_ms,
        animals,
        cluster_ids=cluster_ids,
        kilosort_path=None,
        filter_good_only=True,
        max_saccades_per_animal=max_raster_saccades,
        desc_prefix=f"Raster {name[:15]}",
    )
    n_sacc = sum(
        len(raster_data_by_filter[name][a]) if a in raster_data_by_filter[name] else 0
        for a in animals
    )
    print(f"{name}: animals with raster={len(raster_data_by_filter[name])}, total saccades={n_sacc}")



## Plot and export all figures to PDF (output_folder)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(filtered_collections), 1)))
for idx, (name, df) in enumerate(filtered_collections):
    mean_rate_per_animal = mean_rate_by_filter.get(name, {})
    if not mean_rate_per_animal:
        continue
    # Average rate across animals for this condition (same t_rel for all)
    t_rel = None
    rates = []
    for animal, (t, rate_hz, n_events) in mean_rate_per_animal.items():
        t_rel = t
        rates.append(rate_hz)
    if not rates:
        continue
    mean_rate = np.nanmean(np.stack(rates), axis=0)
    n_total = sum(mean_rate_per_animal[a][2] for a in mean_rate_per_animal)
    ax.plot(t_rel, mean_rate, color=colors[idx % len(colors)], label=f"{name} (n={n_total})", alpha=0.9)
ax.axvline(0, color="k", ls="--", alpha=0.5)
ax.set_xlabel("Time relative to saccade onset (ms)")
ax.set_ylabel("Spike rate (Hz)")
ax.set_title(f"Average pooled spike rate (3 s window, {bin_ms:.0f} ms bins)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_xlim(-half_window_ms, half_window_ms)
ax.grid(True, alpha=0.3)
fig.tight_layout()
out_path = output_folder / "average_rate.pdf"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved {out_path}")



In [ ]:
conditions_with_units = [(name, rate_per_unit_by_filter.get(name, {})) for name, _ in filtered_collections if rate_per_unit_by_filter.get(name, {})]
if not conditions_with_units:
    print("No per-unit data to plot.")
else:
    n_sub = len(conditions_with_units)
    fig, axes = plt.subplots(n_sub, 1, figsize=(10, 3 * n_sub), sharex=True)
    if n_sub == 1:
        axes = [axes]
    for ax_idx, (name, rate_per_animal_per_unit) in enumerate(conditions_with_units):
        ax = axes[ax_idx]
        unit_list = []
        for animal in animals:
            for cid, (t_rel, rate_hz, n_sacc) in rate_per_animal_per_unit.get(animal, {}).items():
                unit_list.append((cid, t_rel, rate_hz))
        seen = set()
        units_plot = [(c, t, r) for c, t, r in unit_list if c not in seen and not seen.add(c)]
        colors = plt.cm.tab20(np.linspace(0, 1, max(len(units_plot), 1)))
        for i, (cid, t_rel, rate_hz) in enumerate(units_plot):
            ax.plot(t_rel, rate_hz, color=colors[i % len(colors)], label=f"Unit {cid}", alpha=0.9)
        ax.axvline(0, color="k", ls="--", alpha=0.5)
        ax.set_ylabel("Spike rate (Hz)")
        ax.set_xlim(-half_window_ms, half_window_ms)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=1)
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{name}", fontsize=10)
        if ax_idx == n_sub - 1:
            ax.set_xlabel("Time relative to saccade onset (ms)")
    fig.suptitle("Average spike rate per unit (by condition)", y=1.01)
    plt.tight_layout()
    out_path = output_folder / "per_unit_rate.pdf"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")



In [ ]:
# Build flat list of (global_row, spike_times_rel) and separator positions (row, label)
raster_rows = []
separator_rows = []  # (row_index, condition_name)
row = 0
for name, df in filtered_collections:
    raster_data_per_animal = raster_data_by_filter.get(name, {})
    if not raster_data_per_animal:
        continue
    # Collect (magnitude, spike_times_rel) per condition (flatten animals)
    entries = []
    for animal in animals:
        saccade_list = raster_data_per_animal.get(animal, [])
        df_animal = df[df["animal"] == animal] if not df.empty else pd.DataFrame()
        for saccade_idx, spike_times_rel in saccade_list:
            mag = np.nan
            if magnitude_col and not df_animal.empty and saccade_idx < len(df_animal):
                mag = df_animal.iloc[saccade_idx].get(magnitude_col, np.nan)
            try:
                mag = float(mag) if (mag == mag) else np.nan
            except (TypeError, ValueError):
                mag = np.nan
            entries.append((mag, spike_times_rel))
    if not entries:
        continue
    entries.sort(key=lambda x: (np.nan if (x[0] != x[0]) else x[0], 0))
    for _, spike_times_rel in entries:
        raster_rows.append((row, spike_times_rel))
        row += 1
    separator_rows.append((row, name))
n_total = len(raster_rows)
if n_total == 0:
    print("No raster data to plot.")
else:
    fig, ax = plt.subplots(figsize=(12, max(6, n_total * 0.06)))
    for row_idx, spike_times_rel in raster_rows:
        if len(spike_times_rel) > 0:
            ax.vlines(spike_times_rel, row_idx - 0.4, row_idx + 0.4, colors="k", linewidths=0.4, alpha=0.6)
    for sep_row, label in separator_rows:
        if 0 < sep_row < n_total:
            ax.axhline(sep_row - 0.5, color="red", ls=":", linewidth=1.2)
            ax.text(-half_window_ms - 80, sep_row - 0.5, label, fontsize=7, va="center", color="red")
    ax.axvline(0, color="r", ls="--", alpha=0.7, linewidth=1.2, label="Saccade onset")
    ax.set_ylabel("Event number (by condition, ordered by magnitude)")
    ax.set_ylim(-0.5, n_total - 0.5)
    ax.set_xlim(-half_window_ms, half_window_ms)
    ax.set_xlabel("Time relative to saccade onset (ms)")
    ax.set_title("Saccade-aligned raster (all conditions, small→large magnitude)")
    ax.grid(True, alpha=0.3, axis="x")
    fig.tight_layout()
    out_path = output_folder / "raster.pdf"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")



In [ ]:
blocks = []
separator_rows_hm = []
global_row = 0
for name, df in filtered_collections:
    per_animal = per_saccade_rate_by_filter.get(name, {})
    if not per_animal:
        continue
    for animal in animals:
        if animal not in per_animal:
            continue
        t_rel_ms, rate_matrix, magnitudes = per_animal[animal]
        n_sacc, n_bins = rate_matrix.shape
        if n_sacc == 0:
            continue
        order = np.argsort(np.where(np.isnan(magnitudes), np.inf, magnitudes))
        rate_ordered = rate_matrix[order]
        blocks.append(rate_ordered)
        global_row += n_sacc
    if blocks:
        separator_rows_hm.append((global_row, name))
if not blocks:
    print("No per-saccade rate data to plot.")
else:
    big = np.vstack(blocks)
    n_total, n_bins = big.shape
    # Debug: print so we can see if data exists (remove or comment out once fixed)
    n_finite = np.sum(np.isfinite(big))
    print(f"[Heatmap debug] shape={big.shape}, dtype={big.dtype}, n_finite={n_finite}, min={np.nanmin(big):.4f}, max={np.nanmax(big):.4f}, mean={np.nanmean(big):.4f}")
    valid = np.isfinite(big) & (big >= 0)
    if np.any(valid):
        vmin = float(np.nanpercentile(big[valid], 2))
        vmax = float(np.nanpercentile(big[valid], 98))
        if vmax <= vmin:
            vmax = vmin + 1.0
        print(f"[Heatmap debug] vmin={vmin:.4f}, vmax={vmax:.4f}")
    else:
        vmin, vmax = 0.0, 1.0
    fig, ax = plt.subplots(figsize=(10, max(5, n_total * 0.04)))
    # Use extent so image fills (left, right, bottom, top); then force axis limits to match
    extent = (-half_window_ms, half_window_ms, n_total, 0)
    im = ax.imshow(big, aspect="auto", extent=extent, interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax, origin="upper")
    # Rasterize the image for PDF export (large arrays can cause PDF rendering issues)
    im.set_rasterized(True)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    # Label first condition at the top (before first separator)
    if separator_rows_hm and separator_rows_hm[0][0] > 0:
        first_sep_row = separator_rows_hm[0][0]
        # Find which condition corresponds to rows 0 to first_sep_row
        first_name = None
        for name, df in filtered_collections:
            per_animal = per_saccade_rate_by_filter.get(name, {})
            if per_animal:
                first_name = name
                break
        if first_name:
            ax.text(-half_window_ms - 80, first_sep_row / 2, first_name, fontsize=9, va="center", color="red", weight="bold")
    # Label separators between conditions
    for sep_row, label in separator_rows_hm:
        if 0 < sep_row < n_total:
            ax.axhline(sep_row, color="red", ls=":", linewidth=1.2)
            ax.text(-half_window_ms - 80, sep_row, label, fontsize=9, va="center", color="red")
    ax.axvline(0, color="white", ls=":", linewidth=1.5)
    ax.set_xlabel("Time relative to saccade onset (ms)")
    ax.set_ylabel("Event number (by condition, ordered by magnitude)")
    ax.set_title("Per-saccade pooled spike rate (all conditions, small→large magnitude)")
    cbar = plt.colorbar(im, ax=ax, label="Spike rate (Hz)")
    fig.tight_layout()
    out_path = output_folder / "heatmap.pdf"
    # For large heatmaps, PDF backend can fail; use PIL/Pillow to convert PNG to PDF
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    # Save as PNG first (this always works)
    png_path = output_folder / "heatmap_temp.png"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor='white', edgecolor='none')
    # Convert PNG to PDF using PIL/Pillow
    try:
        from PIL import Image
        # Increase PIL's decompression bomb limit for large images
        Image.MAX_IMAGE_PIXELS = None  # Disable limit (safe here since we control the source)
        img = Image.open(png_path)
        # Convert RGBA to RGB if needed
        if img.mode == 'RGBA':
            rgb_img = Image.new('RGB', img.size, (255, 255, 255))
            rgb_img.paste(img, mask=img.split()[3])
            img = rgb_img
        # Use lower DPI for PDF (150 is still good quality, reduces file size)
        img.save(out_path, "PDF", resolution=150.0, quality=95)
        png_path.unlink()  # Delete temp PNG
        print(f"Saved {out_path} (converted from PNG at 150 DPI)")
    except ImportError:
        print("Warning: PIL/Pillow not available. Keeping PNG file.")
        png_path.rename(output_folder / "heatmap.png")
        print(f"Saved PNG: {output_folder / 'heatmap.png'}")
        print("Install Pillow (pip install Pillow) to enable PDF export.")
    except Exception as e:
        print(f"Warning: PDF conversion failed: {e}")
        print("Falling back to per-condition PDFs...")
        # Split into one PDF per condition
        try:
            from PIL import Image
            Image.MAX_IMAGE_PIXELS = None
            img = Image.open(png_path)
            if img.mode == 'RGBA':
                rgb_img = Image.new('RGB', img.size, (255, 255, 255))
                rgb_img.paste(img, mask=img.split()[3])
                img = rgb_img
            # Split by condition
            row_start = 0
            for idx, (sep_row, label) in enumerate(separator_rows_hm):
                row_end = sep_row if sep_row < n_total else n_total
                if row_start < row_end:
                    condition_img = img.crop((0, row_start, img.width, row_end))
                    cond_path = output_folder / f"heatmap_{_sanitize(label)}.pdf"
                    condition_img.save(cond_path, "PDF", resolution=150.0, quality=95)
                    print(f"  Saved {cond_path}")
                row_start = row_end
            # Also save first condition if exists
            if separator_rows_hm and separator_rows_hm[0][0] > 0:
                first_name = None
                for name, df in filtered_collections:
                    per_animal = per_saccade_rate_by_filter.get(name, {})
                    if per_animal:
                        first_name = name
                        break
                if first_name:
                    condition_img = img.crop((0, 0, img.width, separator_rows_hm[0][0]))
                    cond_path = output_folder / f"heatmap_{_sanitize(first_name)}.pdf"
                    condition_img.save(cond_path, "PDF", resolution=150.0, quality=95)
                    print(f"  Saved {cond_path}")
            png_path.unlink()
            print(f"Split into {len(separator_rows_hm) + (1 if separator_rows_hm and separator_rows_hm[0][0] > 0 else 0)} condition PDFs")
        except Exception as e2:
            print(f"Split also failed: {e2}")
            png_path.rename(output_folder / "heatmap.png")
            print(f"Saved PNG instead: {output_folder / 'heatmap.png'}")
    plt.close(fig)



## Debug heatmap (optional)

If the heatmap PDF is white or empty, run the cell below. It prints the array shape, min/max, and shows a minimal imshow. If the left panel shows color, the data is fine and the issue is extent/axes; if both panels are white, the data itself is wrong.


In [ ]:
# Rebuild the same stacked array and show diagnostics + minimal imshow (no extent)
_blocks = []
for name, df in filtered_collections:
    per_animal = per_saccade_rate_by_filter.get(name, {})
    for animal in animals:
        if animal not in per_animal:
            continue
        t_rel_ms, rate_matrix, magnitudes = per_animal[animal]
        if rate_matrix.size == 0:
            continue
        order = np.argsort(np.where(np.isnan(magnitudes), np.inf, magnitudes))
        _blocks.append(rate_matrix[order])
if not _blocks:
    print("No blocks to debug.")
else:
    _big = np.vstack(_blocks)
    print("Shape:", _big.shape, "| dtype:", _big.dtype)
    print("Min:", np.nanmin(_big), "| Max:", np.nanmax(_big), "| Mean:", np.nanmean(_big))
    print("Finite count:", np.sum(np.isfinite(_big)), "| NaN count:", np.sum(np.isnan(_big)))
    print("Sample (first row, first 5 bins):", _big[0, :5])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(_big, aspect="auto", cmap="viridis")
    ax1.set_title("Minimal imshow (no extent) - if you see color here, data is OK")
    ax2.imshow(_big, aspect="auto", extent=(-half_window_ms, half_window_ms, _big.shape[0], 0), cmap="viridis", origin="upper")
    ax2.set_xlim(-half_window_ms, half_window_ms)
    ax2.set_ylim(_big.shape[0], 0)
    ax2.set_title("With extent and explicit lims")
    plt.tight_layout()
    _png_path = output_folder / "heatmap_debug.png"
    fig.savefig(_png_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Debug figure also saved to {_png_path} (check if PNG shows data)")



## Combine all exported PDFs into one scrollable PDF

Run this cell after all plots have been exported. It merges the four PDFs (average_rate, per_unit_rate, raster, heatmap) into a single file for sharing.


In [ ]:
try:
    from pypdf import PdfMerger
except ImportError:
    from PyPDF2 import PdfMerger

order = ["average_rate.pdf", "per_unit_rate.pdf", "raster.pdf", "heatmap.pdf"]
pdfs_to_merge = [output_folder / f for f in order if (output_folder / f).exists()]
if not pdfs_to_merge:
    print("No PDFs found in output_folder. Run the plot cells above first.")
else:
    combined_path = output_folder / "spike_rate_pipeline_all_figures.pdf"
    merger = PdfMerger()
    for path in pdfs_to_merge:
        merger.append(str(path))
    merger.write(str(combined_path))
    merger.close()
    print(f"Combined {len(pdfs_to_merge)} PDFs into {combined_path}")

